# Module 2 Assignment: Multi-Table Joins and Aggregates

**Name:**  
**Team:**  
**Team dataset:** `bigquery-public-data.[dataset]`

Copy this file to `members/<your-netid>/M2.ipynb` in your team repo and work there. Submit the finished notebook as a **file upload** on the Module 2 Assignment in Canvas, and push it to your folder.

Three parts, in order. **Part A is your pre-AI attempt: commit and push when Part A is done, before you start Part B** (commit message `M2 Part A: pre-AI attempt`). That commit is your evidence of independent work; do not edit Part A afterwards.

This week every query combines **two or more tables** (or two or more slices of one table) and **summarizes** rows with `GROUP BY`, and every JOIN carries a one- or two-sentence **INNER-versus-LEFT justification**: what happens to the unmatched rows, and why that is the right call for the question. Write and debug SQL in the **BigQuery console** first; here each query lives inside a string passed to `run(...)`.

**Cost guard:** name your columns; never `SELECT *` on a large table. Read the scan estimate in the top-right of the console before you run.

## If your team's dataset is one wide table

Several pairings (Iowa Liquor, Austin 311, USA Names, GA Sample) ship as a single table with the dimensions folded into columns (store, product, category, county...). You still write real JOINs: **derive the dimension from the fact table, then join back to it.** In this module, do it with a CTE (`WITH`), not a saved table:

```sql
WITH stores AS (                       -- one row per store: the dimension you wish you had
  SELECT store_number,
         ANY_VALUE(store_name) AS store_name,   -- ANY_VALUE picks one value per store. A name is only a label.
         ANY_VALUE(county)     AS county        -- county is a grouping key below, so it was CHECKED first (next query)
  FROM `bigquery-public-data.iowa_liquor_sales.sales`
  WHERE date BETWEEN '2024-01-01' AND '2024-12-31'   -- same period as the fact, so labels match the rows
  GROUP BY store_number
),
monthly AS (                           -- the fact, aggregated to the grain you care about
  SELECT store_number, DATE_TRUNC(date, MONTH) AS month, SUM(sale_dollars) AS revenue
  FROM `bigquery-public-data.iowa_liquor_sales.sales`
  WHERE date BETWEEN '2024-01-01' AND '2024-12-31'
  GROUP BY store_number, month
)
SELECT s.county, m.month, SUM(m.revenue) AS county_revenue
FROM monthly AS m
JOIN stores  AS s USING (store_number)
GROUP BY s.county, m.month
ORDER BY county_revenue DESC
LIMIT 20
```

`GROUP BY store_number` is what makes `stores` one row per store, so the join key is unique on that side by construction. What it does **not** guarantee is that an attribute you then group by, like `county`, has a single value per store; `ANY_VALUE` would silently pick one. So check the attribute separately before you trust it. This is the check behind the `county` comment above; it uses `HAVING`, which is in scope this week:

```sql
-- Stores whose county is not stable in 2024. Verified result: 0 rows (2,162 stores), so county is safe to GROUP BY.
-- The same check on store_name returns 13 stores with two spellings, which is why the name is treated as a label only.
SELECT store_number, COUNT(DISTINCT county) AS n_counties
FROM `bigquery-public-data.iowa_liquor_sales.sales`
WHERE date BETWEEN '2024-01-01' AND '2024-12-31'
GROUP BY store_number
HAVING COUNT(DISTINCT county) > 1
```

**A second pattern that needs no second table at all: join two slices of the same table.** "Which stores grew year over year, and which have no prior-year baseline?" is a LEFT-versus-INNER question in its purest form:

```sql
WITH y2023 AS (
  SELECT store_number, SUM(sale_dollars) AS revenue_2023
  FROM `bigquery-public-data.iowa_liquor_sales.sales`
  WHERE date BETWEEN '2023-01-01' AND '2023-12-31'
  GROUP BY store_number
),
y2024 AS (
  SELECT store_number, ANY_VALUE(store_name) AS store_name, SUM(sale_dollars) AS revenue_2024
  FROM `bigquery-public-data.iowa_liquor_sales.sales`
  WHERE date BETWEEN '2024-01-01' AND '2024-12-31'
  GROUP BY store_number
)
SELECT a.store_number, a.store_name, a.revenue_2024, b.revenue_2023,
       a.revenue_2024 - b.revenue_2023 AS change,
       b.store_number IS NULL AS no_2023_baseline    -- TRUE only for rows a LEFT JOIN kept and an INNER JOIN would drop
FROM y2024 AS a
LEFT JOIN y2023 AS b USING (store_number)
ORDER BY change DESC NULLS LAST                      -- growers first; the no-baseline stores have NULL change and sort last
LIMIT 20
```

That answers "which stores grew". The second half of the question, "which have no baseline", is exactly the set of rows an `INNER JOIN` would have thrown away, so show them on their own. A `WITH` clause belongs to one statement only, so the CTEs are repeated here:

```sql
-- Stores selling in 2024 with no 2023 sales: the rows the LEFT JOIN kept.
-- Verified: 163 such stores; this shows the top 20 of them by 2024 revenue.
WITH y2023 AS (
  SELECT store_number
  FROM `bigquery-public-data.iowa_liquor_sales.sales`
  WHERE date BETWEEN '2023-01-01' AND '2023-12-31'
  GROUP BY store_number
),
y2024 AS (
  SELECT store_number, ANY_VALUE(store_name) AS store_name, SUM(sale_dollars) AS revenue_2024
  FROM `bigquery-public-data.iowa_liquor_sales.sales`
  WHERE date BETWEEN '2024-01-01' AND '2024-12-31'
  GROUP BY store_number
)
SELECT a.store_number, a.store_name, a.revenue_2024
FROM y2024 AS a
LEFT JOIN y2023 AS b USING (store_number)
WHERE b.store_number IS NULL
ORDER BY a.revenue_2024 DESC
LIMIT 20
```

Run the count check on this one: 2,162 stores sold something in 2024; an `INNER JOIN` keeps 1,999 of them, a `LEFT JOIN` keeps all 2,162. The 163 that differ are the stores the second query lists (top 20 shown). Whether they belong in the answer depends on the question, which is the justification you write.

Pick **one** pattern that fits your stakeholder question; the two examples are illustrations, not extra work to complete. Both practice the same join mechanics, join-type choice and row-count check as two physical tables, with one honest difference: splitting one source and rejoining it adds no new information, whereas a real second table can. What it does teach is grain control, and you still have to check that your derived key is unique on the one side. These derived dimensions are the tables you will formalize in Modules 4 to 6. Teams whose dataset is already multi-table (Stack Overflow, Olist, NYC Taxi + `taxi_zone_geom`) join the tables you have.

## Part A — Pre-AI attempt (no AI yet)

**Stakeholder question:** [one sentence: who is asking, and what do they want to know?]

**Tables (or CTEs) involved:** [table 1] joined to [table 2] on [key]

Write your own first try at the queries below, by hand. Wrong output and errors are fine and expected; Part A is graded on effort and reasoning, not correctness. **Above every JOIN, write one line predicting what happens to the row count and which rows could drop**; you will check it against the real numbers in Part B.

When you finish Part A: **Kernel → Restart & Run All, save, `git add`, `git commit -m "M2 Part A: pre-AI attempt"`, `git push`.** Then continue to Part B without changing these cells.

In [ ]:
# Setup: run once. Uses the personal Google account from the BigQuery Sandbox page.
# See Jupyter Environment Setup → "Connect Your Notebook to BigQuery".
import pandas_gbq
PROJECT_ID = "your-sandbox-project-id"   # from the BigQuery console, top-left project selector

def run(sql):
    """Run a SQL string against BigQuery and return a DataFrame."""
    return pandas_gbq.read_gbq(sql, project_id=PROJECT_ID)


In [ ]:
# A0 — Row counts BEFORE any join. Count each side you are about to join; keep these numbers.
run("""
SELECT
  (SELECT COUNT(*) FROM `bigquery-public-data.[dataset].[table_1]`) AS table_1_rows,
  (SELECT COUNT(*) FROM `bigquery-public-data.[dataset].[table_2]`) AS table_2_rows
""")


In [ ]:
# A1 — Question: [what question does this answer?]
# JOIN prediction: [row count will go up / down / stay the same, because ...; the rows that could drop are ...]
# JOIN type: [INNER / LEFT] because [what happens to unmatched rows, and why that is right for this question]
run("""
SELECT
  [columns],
  COUNT(*) AS n
FROM `bigquery-public-data.[dataset].[table_1]` AS a
[INNER | LEFT] JOIN `bigquery-public-data.[dataset].[table_2]` AS b
  ON a.[key] = b.[key]
WHERE [condition]
GROUP BY [columns]
ORDER BY n DESC
LIMIT 20
""")


In [ ]:
# A2 — Question: [ ... ]
# JOIN prediction: [ ... ]
# JOIN type: [INNER / LEFT] because [ ... ]
run("""

""")


In [ ]:
# A3 — Question: [ ... ]
# JOIN prediction: [ ... ]
# JOIN type: [INNER / LEFT] because [ ... ]
run("""

""")


## Part B — Queries with AI (3 to 5, multi-table + aggregate)

Now bring AI in: ask for what you actually want, read what it gives you, keep / change / reject, and run it. One cell per query. The comment above each query states the question it answers **and** the INNER-versus-LEFT justification (or put the justification in a short Markdown cell beside the query). Use `GROUP BY`, and `HAVING` where the question filters on an aggregate.

**Run the check from the worked example, for every JOIN you submit.** Count the rows before the JOIN and after it, and record both numbers in the `# Row check:` line above the query, one line per JOIN if a query has more than one (the B1 check cell shows one way to get them; for a query that joins CTEs, count the CTE you join from). If the number went **down**, find out which rows left and decide, on purpose, whether they should have. If it went **up**, one side has more than one row per key: ask whether that many-side grain is the grain you meant. It is fine when the question is at that grain; it is double-counting the moment you `SUM` a measure that belongs to the one side. Compare the real numbers with your Part A predictions. An AI will not do this for you, and neither will an error message.

In [ ]:
# B1 — Question: [ ... ]
# JOIN type: [INNER / LEFT] because [ ... ]
# Row check: before = [ n ], after = [ n ]. Which rows changed, and should they have? [ ... ]
run("""

""")


In [ ]:
# B1 check — rows before vs after the join, with the SAME a-side filter on both sides, so any difference is the JOIN's doing.
# A filter on b's columns cannot exist before the join; if B1 has one, it shows up only in after_filters.
# If B1 joins CTEs (WITH ...), count the CTE you join from instead of the raw table.
run("""
SELECT
  (SELECT COUNT(*)
     FROM `bigquery-public-data.[dataset].[table_1]` AS a
    WHERE [B1's a-side filter, or TRUE]) AS before_join,
  (SELECT COUNT(*)
     FROM `bigquery-public-data.[dataset].[table_1]` AS a
     [INNER | LEFT] JOIN `bigquery-public-data.[dataset].[table_2]` AS b ON a.[key] = b.[key]
    WHERE [B1's a-side filter, or TRUE]) AS after_join,
  (SELECT COUNT(*)
     FROM `bigquery-public-data.[dataset].[table_1]` AS a
     [INNER | LEFT] JOIN `bigquery-public-data.[dataset].[table_2]` AS b ON a.[key] = b.[key]
    WHERE [B1's full WHERE clause, or TRUE]) AS after_filters
""")


In [ ]:
# B2 — Question: [ ... ]
# JOIN type: [INNER / LEFT] because [ ... ]
# Row check: before = [ n ], after = [ n ]. Which rows changed, and should they have? [ ... ]
run("""

""")


In [ ]:
# B3 — Question: [ ... ]
# JOIN type: [INNER / LEFT] because [ ... ]
# Row check: before = [ n ], after = [ n ]. Which rows changed, and should they have? [ ... ]
run("""

""")


In [ ]:
# B4 (optional) — Question: [ ... ]
# JOIN type: [INNER / LEFT] because [ ... ]
# Row check: before = [ n ], after = [ n ]. Which rows changed, and should they have? [ ... ]
run("""

""")


In [ ]:
# B5 (optional) — Question: [ ... ]
# JOIN type: [INNER / LEFT] because [ ... ]
# Row check: before = [ n ], after = [ n ]. Which rows changed, and should they have? [ ... ]
run("""

""")


**Prediction check:** [For each JOIN in Part A: what you predicted, what the real before/after counts were, and what explains any difference. Two or three sentences.]

## Part C — AI Attribution Log

One row per AI-mediated step that changed what you shipped (decisions, not keystrokes). "It looked right" is not a verification; a row-count check is.

| # | Where (which query) | Tool (and model if known) | What I asked for | What it gave me | What I did with it (accepted / edited / rejected) | How I verified it |
|---|---|---|---|---|---|---|
| 1 |  |  |  |  |  |  |
| 2 |  |  |  |  |  |  |

## Reflection

[One specific surprise from writing this module's queries, and why it was unexpected. Three to five sentences. A JOIN that changed the row count in a way you did not predict is a good candidate.]

## Before submitting

- Every JOIN has a justification (INNER or LEFT, and why) and a filled-in `# Row check:` line with the before and after counts.
- Kernel → Restart & Run All; every cell shows the output you intend the grader to see.
- Save, `git add`, `git commit -m "M2 complete"`, `git push` (ungraded in Modules 1–2; required from Module 3).
- Upload this `.ipynb` on the Module 2 Assignment page in Canvas.